In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df= pd.read_csv ("USvideos.csv")

print("Dataset shape:", df.shape)

display (df.head())

df.info()


In [ ]:
category_counts =df["category_id"].value_counts().sort_index()

display(category_counts)


In [ ]:

category_map = {
    1: "Film & Animation",
    2: "Autos & Vehicles",
    10: "Music",
    15: "Pets & Animals",
    17: "Sports",
    19: "Travel & Events",
    20: "Gaming",
    22: "People & Blogs",
    23: "Comedy",
    24: "Entertainment",
    25: "News & Politics",
    26: "Howto & Style",
    27: "Education",
    28: "Science & Technology",
    29: "Nonprofits & Activism",
    43: "Shows"
}


df["category_name"] = df["category_id"].map(category_map)


display(df["category_name"].value_counts(dropna=False))


In [ ]:

entertainment_categories = [
    "Film & Animation",
    "Entertainment",
    "Music",
    "Comedy",
    "Shows"
]


entertainment_df = df[
    df["category_name"].isin(entertainment_categories)
].copy()


print("Entertainment records:", len(entertainment_df))
print("Unique videos:", entertainment_df["video_id"].nunique())

display(entertainment_df["category_name"].value_counts())


In [ ]:

total_records = len(entertainment_df)


unique_videos = entertainment_df["video_id"].nunique()


repeated_records = total_records - unique_videos


print("Total records:", total_records)
print("Unique videos:", unique_videos)
print("Repeated appearances:", repeated_records)


In [ ]:
entertainment_df["trending_date"] = pd.to_datetime(
    entertainment_df["trending_date"],
    format="%y.%d.%m")


In [ ]:

entertainment_df = entertainment_df.sort_values("trending_date")

unique_entertainment_df = entertainment_df.drop_duplicates(
    subset="video_id",
    keep="last"
).copy()

print("Rows remaining:", len(unique_entertainment_df))
print("Unique videos:", unique_entertainment_df["video_id"].nunique())


In [ ]:

category_distribution = unique_entertainment_df["category_name"].value_counts()

display(category_distribution)


In [ ]:

category_percentages = (
    unique_entertainment_df["category_name"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(category_percentages)


In [ ]:

sample_sizes = {
    "Entertainment": 74,
    "Music": 37,
    "Comedy": 25,
    "Film & Animation": 14
}

sample_df = pd.concat([
    unique_entertainment_df[
        unique_entertainment_df["category_name"] == category
    ].sample(n=count, random_state=42)
    for category, count in sample_sizes.items()
], ignore_index=True)

print("Total sampled videos:", len(sample_df))
print("Unique sampled videos:", sample_df["video_id"].nunique())

display(sample_df["category_name"].value_counts())



In [ ]:

tagging_preview = sample_df[
    ["video_id", "category_name", "title", "description"]
]

display(tagging_preview.head(15))


In [ ]:

random_preview = sample_df.sample(
    n=15,
    random_state=10
)

display(random_preview[["category_name", "title"]])


In [ ]:

tagging_df = sample_df[
    ["video_id", "category_name", "title",
     "channel_title", "description", "views",
     "likes", "comment_count"]
].copy()

tagging_df["video_type"] = ""
tagging_df["content_purpose"] = ""
tagging_df["content_topic"] = ""

display(tagging_df.head())


In [ ]:
analysis_df = pd.read_excel(
    "youtube_content_tagging_reviewed.xlsx",
    sheet_name=0
)

print(analysis_df.shape)
analysis_df.head()

In [ ]:

print(analysis_df[["video_type", "content_purpose", "content_topic"]].isna().sum())


In [ ]:

analysis_df[["views", "likes", "comment_count"]].describe()


In [ ]:

analysis_df["total_interactions"] = (
    analysis_df["likes"] + analysis_df["comment_count"]
)

analysis_df[["title", "likes", "comment_count", "total_interactions"]].head()


In [ ]:

analysis_df["interaction_to_view_rate"] = (
    analysis_df["total_interactions"] / analysis_df["views"]
) * 100

analysis_df[["title", "views", "total_interactions", "interaction_to_view_rate"]].head()


In [ ]:

print("Videos with zero views:", (analysis_df["views"] == 0).sum())

print("Videos with missing views:", analysis_df["views"].isna().sum())

print("Videos with missing likes:", analysis_df["likes"].isna().sum())

print("Videos with missing comments:", analysis_df["comment_count"].isna().sum())

print("Videos with negative interactions:", (analysis_df["total_interactions"] < 0).sum())


In [ ]:

analysis_df.columns.tolist()


In [ ]:

video_type_performance = (
    analysis_df.groupby("video_type")
    .agg(
        video_count=("video_id", "count"),
        median_views=("views", "median"),
        median_interactions=("total_interactions", "median"),
        median_interaction_rate=("interaction_to_view_rate", "median")
    )
    .sort_values("median_interaction_rate", ascending=False)
)

video_type_performance.round(2)


In [ ]:

content_purpose_performance = (
    analysis_df.groupby("content_purpose")
    .agg(
        video_count=("video_id", "count"),
        median_views=("views", "median"),
        median_interactions=("total_interactions", "median"),
        median_interaction_rate=("interaction_to_view_rate", "median")
    )
    .sort_values("median_interaction_rate", ascending=False)
)

content_purpose_performance.round(2)


In [ ]:

analysis_df["content_purpose"].value_counts(dropna=False)


In [ ]:

content_topic_performance = (
    analysis_df.groupby("content_topic")
    .agg(
        video_count=("video_id", "count"),
        median_views=("views", "median"),
        median_interactions=("total_interactions", "median"),
        median_interaction_rate=("interaction_to_view_rate", "median")
    )
    .sort_values("median_interaction_rate", ascending=False)
)

content_topic_performance.round(2)


In [ ]:

purpose_topic_performance = (
    analysis_df.groupby(["content_purpose", "content_topic"])
    .agg(
        video_count=("video_id", "count"),
        median_interaction_rate=("interaction_to_view_rate", "median")
    )
    .reset_index()
    .sort_values("median_interaction_rate", ascending=False)
)

purpose_topic_performance.round(2)


In [ ]:

filtered_performance = purpose_topic_performance[
    purpose_topic_performance["video_count"] >= 10
]

filtered_performance.round(2)


In [ ]:

print("Number of groups:", len(filtered_performance))

print("Videos represented:", filtered_performance["video_count"].sum())


In [ ]:

analysis_df.to_csv(
    "youtube_engagement_analysis.csv",
    index=False
)

print("CSV exported successfully!")


In [ ]:

export_check = pd.read_csv("youtube_engagement_analysis.csv")

print(export_check.shape)
export_check.head()


In [ ]:

import pandas as pd

analysis_df = pd.read_excel(
    "youtube_content_tagging_reviewed.xlsx",
    sheet_name=0
)

print(analysis_df.shape)


In [ ]:

print("Total rows:", len(analysis_df))

print("Unique video IDs:", analysis_df["video_id"].nunique())

print("Duplicate video IDs:", analysis_df["video_id"].duplicated().sum())


In [ ]:

duplicate_videos = analysis_df[
    analysis_df["video_id"].duplicated(keep=False)
]

missing_id_videos = analysis_df[
    analysis_df["video_id"].isna()
]

display(duplicate_videos)

display(missing_id_videos)


In [ ]:

original_df = pd.read_csv("USvideos.csv")

missing_titles = analysis_df.loc[
    analysis_df["video_id"].isna(), "title"
]

matches = original_df[
    original_df["title"].isin(missing_titles)
]

display(
    matches[
        ["video_id", "title", "channel_title"]
    ].drop_duplicates()
)


In [ ]:

janelle_matches = original_df[
    original_df["title"].str.contains(
        "I Like That",
        case=False,
        na=False,
        regex=False
    )
]

display(
    janelle_matches[
        ["video_id", "title", "channel_title"]
    ].drop_duplicates()
)


In [ ]:

analysis_df.loc[83, "video_id"] = "-WedqN1YbM"

analysis_df.loc[94, "video_id"] = "-xAMnJgc3oA"

print("Total rows:", len(analysis_df))
print("Unique video IDs:", analysis_df["video_id"].nunique())
print("Missing video IDs:", analysis_df["video_id"].isna().sum())
print("Duplicate video IDs:", analysis_df["video_id"].duplicated().sum())


In [ ]:

analysis_df["total_interactions"] = (
    analysis_df["likes"] + analysis_df["comment_count"]
)

analysis_df["interaction_to_view_rate"] = (
    analysis_df["total_interactions"] / analysis_df["views"]
) * 100

analysis_df.to_csv(
    "youtube_engagement_analysis.csv",
    index=False
)

print("Corrected CSV exported!")
print("Final dataset shape:", analysis_df.shape)
